# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
Joaquín Venezia. LU: 1338/23
Simón Atías. LU: 526/23
Lucía Insua. LU: 580/22
Victoria Arce. LU: 553/23

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

**Respuesta 1.a**

* **Rango:** $\mathrm{rank}(Q)=11$. Como $Q$ tiene 12 columnas, $\dim\ker Q = 12-11 = 1$, es decir, las soluciones no nulas de $Q\mathbf{m}=\mathbf{0}$ son $\mathbf{m}$ y sus múltiplos (única a menos de un factor de escala).
* **Cantidad mínima de puntos:** cada punto aporta 2 filas, y se necesitan al menos 11 filas independientes: $2n \ge 11 \Rightarrow n \ge 6$. Con 5 puntos hay solo 10 ecuaciones, que no alcanzan.
* **Condición:** contar filas es necesario pero no suficiente: las $2n$ filas deben tener rango 11, y eso depende de la posición de los puntos. Los puntos no pueden ser todos coplanares; deben estar en posición general (tampoco sobre una superficie crítica, como una cúbica alabeada que contenga al centro de la cámara).
  * *Caso coplanar.* Si todos los puntos cumplen $aU+bV+cW+d=0$, el vector $\boldsymbol{\pi}=(a,b,c,d)^T$ satisface $P_i^T\boldsymbol{\pi}=0$ para todo $i$. Entonces $(\boldsymbol{\pi},\mathbf{0},\mathbf{0})$, $(\mathbf{0},\boldsymbol{\pi},\mathbf{0})$ y $(\mathbf{0},\mathbf{0},\boldsymbol{\pi})$ pertenecen a $\ker Q$, que ya tiene dimensión $\ge 3$; sumando la solución verdadera $\mathbf{m}$ queda $\dim\ker Q\ge 4$ y $\mathrm{rank}(Q)\le 8$. Por ejemplo, con $W=0$ las tres columnas asociadas a $W$ son nulas (lo que solo da $\mathrm{rank}(Q)\le 9$) y el 8 aparece al agregar la solución verdadera. La solución deja de ser única.
* **Con datos reales:** el rango 11 vale para datos exactos. Con ruido de medición, $Q$ tiene rango numérico 12 (en `calib/3d` el menor valor singular es $\sigma_{12}\approx 0{,}008>0$) y no existe $\mathbf{m}\neq\mathbf{0}$ con $Q\mathbf{m}=\mathbf{0}$ exacto; por eso se resuelve la minimización de 1.b.

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 8.

**Respuesta 1.b**

Se busca $\min_{\|\mathbf{m}\|=1} \|Q\mathbf{m}\|^2$. Reescribiendo:

$$\|Q\mathbf{m}\|^2 = (Q\mathbf{m})^T(Q\mathbf{m}) = \mathbf{m}^T Q^T Q\,\mathbf{m} = \mathbf{m}^T S\,\mathbf{m}, \qquad S = Q^TQ.$$

**Propiedades de $S$.** Es simétrica, ya que $S^T = Q^T(Q^T)^T = S$, y semidefinida positiva, ya que $\mathbf{x}^TS\mathbf{x} = \|Q\mathbf{x}\|^2 \ge 0$. Por el teorema espectral, $S = V\Lambda V^T$, con $V$ ortogonal (columnas $\mathbf{v}_1,\dots,\mathbf{v}_{12}$ autovectores ortonormales) y $\Lambda = \mathrm{diag}(\lambda_1\ge\dots\ge\lambda_{12}\ge 0)$.

**Cambio de base.** Como las $\mathbf{v}_i$ son una base ortonormal, $\mathbf{m} = V\mathbf{c} = \sum_i c_i\mathbf{v}_i$. Al ser $V$ ortogonal conserva la norma:

$$\|\mathbf{m}\|^2 = \|\mathbf{c}\|^2 = \sum_i c_i^2 = 1, \qquad \mathbf{m}^TS\mathbf{m} = \mathbf{c}^T\Lambda\,\mathbf{c} = \sum_i \lambda_i c_i^2.$$

**Minimización.** Los pesos $c_i^2\ge 0$ suman 1, así que $\sum_i\lambda_i c_i^2$ es un promedio ponderado de los autovalores y no puede ser menor que el menor de ellos:

$$\sum_i \lambda_i c_i^2 \;\ge\; \lambda_{12}\sum_i c_i^2 = \lambda_{12}.$$

La igualdad se alcanza con $\mathbf{c} = \mathbf{e}_{12}$ (todo el peso en el menor autovalor), es decir, $\mathbf{m} = V\mathbf{e}_{12} = \mathbf{v}_{12}$. La solución es única (salvo signo) cuando $\lambda_{12}<\lambda_{11}$, es decir, cuando el menor autovalor es simple. Eso ocurre si $\mathrm{rank}(Q)=11$ con datos en posición general (1.a); si el menor autovalor fuera múltiple, cualquier vector unitario de ese autoespacio sería solución.

**Resultado.** $\mathbf{m}$ es el autovector de $Q^TQ$ asociado a su menor autovalor. Si $Q = U\Sigma V^T$ es su SVD, entonces $Q^TQ = V\Sigma^2V^T$, por lo que $\mathbf{m}$ es la última columna de $V$ (vector singular derecho asociado a $\sigma_{\min}$), y el valor mínimo es $\|Q\mathbf{m}\| = \sigma_{\min}$.

Observaciones: $\mathbf{m}$ queda definido a menos de signo (si $\mathbf{m}$ es solución, $-\mathbf{m}$ también), que se fija con un criterio físico (por ejemplo, que los puntos queden delante de la cámara). Con datos ideales $\sigma_{\min}=0$ y $Q\mathbf{m}=\mathbf{0}$ exacto; con ruido $\sigma_{\min}>0$ y $\mathbf{m}$ es la mejor solución en el sentido de mínimos cuadrados homogéneos. Luego $\mathbf{m}$ se reordena en la matriz de proyección $M\in\mathbb{R}^{3\times4}$.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
import os
import sys
import numpy as np
sys.path.insert(0, "../..")   # modulo alc
sys.path.insert(0, "..")      # calibracion.py
from calibracion import (obtenerQ3d, resolverDLT3D,
                         devolverMatricesCalibracion3d, verificarCalibracion3d)

np.set_printoptions(precision=3, suppress=True)

# Carpeta de datos: en el zip del template aparece como " calib" (con un espacio adelante)
nombre = [d for d in os.listdir(".") if d.strip() == "calib"][0]
carpeta = os.path.join(nombre, "3d")

# 12 puntos 3D (U, V, W) y 210 detecciones de esos mismos puntos: cada fila
# tiene los pares (u, v) intercalados (24 columnas).
objPts = np.loadtxt(os.path.join(carpeta, "p_W_corners.txt"), delimiter=",")   # 12 x 3
detecciones = np.loadtxt(os.path.join(carpeta, "detected_corners.txt"))        # 210 x 24

variantes = [("libre",            False, False),
             ("alfa_u = alfa_v",  True,  False),
             ("gamma = 0",        False, True),
             ("ambas",            True,  True)]

for fila in [0, 50, 100]:
    imgPts = detecciones[fila].reshape(12, 2)

    # DLT: Q (24 x 12) -> M (3 x 4)
    Q = obtenerQ3d(imgPts, objPts)
    M = resolverDLT3D(Q)

    print(f"===== Deteccion {fila} =====")
    for nombreVar, alfasIguales, gammaCero in variantes:
        R, t, K = devolverMatricesCalibracion3d(M, alfasIguales, gammaCero)
        chequeo = verificarCalibracion3d(M, R, t, K, imgPts, objPts)
        print(f"--- {nombreVar} ---")
        print("K =\n", K)
        print("R =\n", R)
        print("t =", t)
        print(f"error maximo de reproyeccion: {chequeo['error_reproyeccion']:.2f} px")
        if nombreVar == "libre":
            print(f"max|R R^T - I| = {chequeo['ortogonalidad']:.1e}, "
                  f"det(R) = {chequeo['det_R']:.3f}, "
                  f"puntos delante de la camara: {chequeo['delante_camara']}")


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.


**Respuesta 3 (rango de $Q$).** $Q\in\mathbb{R}^{2n\times 9}$ y $\mathbf{h}$ está definido a menos de escala (8 grados de libertad), así que hace falta $\mathrm{rank}(Q)=8$: así $\dim\ker Q = 9-8=1$ y la solución es única salvo factor de escala. Cada punto aporta 2 ecuaciones, luego $n\ge 4$, y no puede haber 3 puntos colineales (ni en el tablero ni en la imagen). Acá $n=77$, así que $Q$ es de $154\times 9$.

**Cómo se obtienen las esquinas.** La carpeta no trae las coordenadas $(u,v)$, así que se detectan con código propio (`tp/esquinas.py`): las esquinas internas son puntos silla ($\det$ Hessiana de la imagen suavizada muy negativo); se buscan máximos locales con precisión subpíxel y se ordenan armando la grilla y quedándose con el bloque completo de $11\times 7$ centrado.

**Algoritmo.** (1) $\mathbf{H}_j$ por DLT con normalización de Hartley, (2) $A\mathbf{b}=0$ con 2 filas por vista, (3) $B=K^{-T}K^{-1}$ y Cholesky, (4) pose de cada vista con $\mathbf r_1=\lambda K^{-1}\mathbf c_1$, $\mathbf r_2=\lambda K^{-1}\mathbf c_2$, $\mathbf r_3=\mathbf r_1\times\mathbf r_2$, $\mathbf t=\lambda K^{-1}\mathbf c_3$.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
sys.path.insert(0, "../..")   # modulo alc
sys.path.insert(0, "..")      # calibracion.py y esquinas.py
from calibracion import resolverH, armarA, calcularMatrizK2D, obtenerPose2D
from esquinas import esquinasTablero
from alc import multiplicacionDeMatrices, traspuesta

np.set_printoptions(precision=3, suppress=True)

# Puntos del tablero (mm): (U, V) = (30 i, 30 j), i = 0..10, j = 0..6, W = 0.
# Orden k = j*11 + i, igual al de las esquinas detectadas.
objPts = np.array([[30.0 * i, 30.0 * j] for j in range(7) for i in range(11)])

# Carpeta de datos: en el zip del template aparece como " calib" (con un espacio adelante)
nombre = [d for d in os.listdir(".") if d.strip() == "calib"][0]
carpeta = os.path.join(nombre, "coplanar")

# 1) Esquinas (u, v) de cada vista y homografia H_j por DLT (con normalizacion de Hartley)
Hlst = []
imgPtsLst = []
for k in range(1, 21):
    imagen = mpimg.imread(os.path.join(carpeta, f"Im_R_{k}.png"))
    imgPts = esquinasTablero(imagen)                          # 77 x 2
    imgPtsLst.append(imgPts)
    Hlst.append(resolverH(imgPts, objPts))

# 2) K a partir de las 20 homografias
A = armarA(Hlst)                  # 40 x 6
K = calcularMatrizK2D(A)
print("K =\n", K)

# 3) Pose de las vistas 1, 10 y 20 y error de reproyeccion
for k in [1, 10, 20]:
    R, t = obtenerPose2D(Hlst[k - 1], K)
    Rt = np.hstack([R, t.reshape(3, 1)])
    P = np.hstack([objPts, np.zeros((77, 1)), np.ones((77, 1))])   # (U, V, 0, 1)
    proy = multiplicacionDeMatrices(P, traspuesta(multiplicacionDeMatrices(K, Rt)))
    uv = proy[:, 0:2] / proy[:, 2:3]
    err = np.sqrt(np.sum((uv - imgPtsLst[k - 1]) ** 2, axis=1))
    print(f"\nIm_R_{k}: R =\n{R}\n t (mm) = {t}\n error de reproyeccion: medio {np.mean(err):.2f} px, max {np.max(err):.2f} px")


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---